In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("/content/csv_result-NIMS.csv")
df.head(10)

,id,min_fpktl,mean_fpktl,max_fpktl,std_fpktl,min_bpktl,mean_bpktl,max_bpktl,std_bpktl,min_fiat,...,mean_biat,max_biat,std_biat,duration,proto,total_fpackets,total_fvolume,total_bpackets,total_bvolume,class
0,1,40,45,50,3,40,48,113,18,361,...,20317,58785,20354,7241033,6,12,546,15,727,TELNET
1,2,40,46,56,4,40,60,98,17,324,...,16003,129174,33664,240455,6,12,553,16,975,FTP
2,3,52,226,460,207,52,890,1500,607,152,...,15731,90358,27594,6870746,6,14,3166,14,12473,HTTP
3,4,56,62,64,4,76,170,213,64,235202,...,5723,5723,0,78872373,17,4,248,4,683,DNS
4,5,101,101,101,0,195,412,634,115,1528,...,65386,428798,160266,262790503,17,21,2121,21,8659,lime
5,6,52,225,756,248,52,231,804,254,136,...,42274,154743,44619,681767,6,18,4062,17,3933,localForwarding
6,7,52,8222,16436,7849,52,62,756,41,2,...,14723,139626,25909,9998248,6,1062,8731838,680,42677,remoteForwarding
7,8,52,1469,1500,183,52,56,756,32,6,...,5484,830068,30431,5754219,6,2512,3691910,1049,59401,scp
8,9,52,1450,1500,245,52,64,756,54,6,...,13943,404559,47655,5312613,6,692,1003838,381,24701,sftp
9,10,52,146,692,178,52,157,756,172,148,...,79573,384275,102164,5168046,6,31,4530,26,4097,x11


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 713851 entries, 0 to 713850
Data columns (total 24 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   id              713851 non-null  int64 
 1   min_fpktl       713851 non-null  int64 
 2   mean_fpktl      713851 non-null  int64 
 3   max_fpktl       713851 non-null  int64 
 4   std_fpktl       713851 non-null  int64 
 5   min_bpktl       713851 non-null  int64 
 6   mean_bpktl      713851 non-null  int64 
 7   max_bpktl       713851 non-null  int64 
 8   std_bpktl       713851 non-null  int64 
 9   min_fiat        713851 non-null  int64 
 10  mean_fiat       713851 non-null  int64 
 11  max_fiat        713851 non-null  int64 
 12  std_fiat        713851 non-null  int64 
 13  min_biat        713851 non-null  int64 
 14  mean_biat       713851 non-null  int64 
 15  max_biat        713851 non-null  int64 
 16  std_biat        713851 non-null  int64 
 17  duration        713851 non-nu

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# Classifiers
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

# 1. Drop rows where target variable is missing
features = df.copy()
if 'id' in features.columns:
    features = features.drop(columns=['id'])

# Separate features and target
X = features.drop(columns=['class'])
y = features['class']

# Identify numeric columns for scaling
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Fill any missing values
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

# Encode target labels to integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Scale numerical features
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])

# Convert categorical features if any remain
X_encoded = pd.get_dummies(X_scaled)

# 2. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (571080, 22)
Testing shape: (142771, 22)


In [5]:
# Initialize dictionaries to hold models and their performances
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=12, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1, eval_metric='mlogloss'),
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42, n_jobs=-1),
    'Linear SVM': LinearSVC(dual=False, random_state=42, max_iter=1000),
    # A single-layer MLP to balance speed and accuracy on large datasets
    'Neural Network': MLPClassifier(hidden_layer_sizes=(64,), max_iter=200, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results[name] = {
            'model': model,
            'accuracy': acc,
            'predictions': y_pred
        }
        print(f"{name} completed with Accuracy: {acc:.4f}\n")
    except Exception as e:
        print(f"Failed to train {name}: {e}\n")

Training Decision Tree...
Decision Tree completed with Accuracy: 0.9983

Training Random Forest...
Random Forest completed with Accuracy: 0.9983

Training XGBoost...
XGBoost completed with Accuracy: 0.9990

Training Logistic Regression...
Logistic Regression completed with Accuracy: 0.9449

Training Linear SVM...


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Linear SVM completed with Accuracy: 0.9399

Training Neural Network...
Neural Network completed with Accuracy: 0.9951



In [6]:
# Detailed classification report comparison
for name, info in results.items():
    print(f"=== {name} Classification Report ===")
    print(classification_report(y_test, info['predictions'], target_names=label_encoder.classes_))
    print("\n")

=== Decision Tree Classification Report ===
                  precision    recall  f1-score   support

             DNS       1.00      0.98      0.99      7603
             FTP       1.00      1.00      1.00       346
            HTTP       0.99      1.00      0.99      2381
          TELNET       1.00      1.00      1.00       250
            lime       1.00      1.00      1.00    129255
 localForwarding       1.00      0.99      1.00       512
remoteForwarding       0.99      1.00      0.99       484
             scp       1.00      0.99      0.99       489
            sftp       1.00      0.99      0.99       482
           shell       0.96      1.00      0.98       498
             x11       1.00      0.99      0.99       471

        accuracy                           1.00    142771
       macro avg       0.99      0.99      0.99    142771
    weighted avg       1.00      1.00      1.00    142771



=== Random Forest Classification Report ===
                  precision    recall

In [8]:
test = pd.read_csv("/content/traffic_features.csv")
test['class'].unique()

array(['SSH-localForward', 'HTTPS', 'FTPS', 'DoT', 'SSH-remoteForward',
       'SFTP', 'SSH-x11', 'TLS-Telnet', 'Encrypted-P2P', 'SSH-shell',
       'SCP'], dtype=object)

In [9]:
# Let's check the unique classes from the training dataset vs. the test dataset
train_classes = sorted(df['class'].unique())
test_classes = sorted(test['class'].unique())

print("Train classes (2000):", train_classes)
print("Test classes (2026):", test_classes)

Train classes (2000): ['DNS', 'FTP', 'HTTP', 'TELNET', 'lime', 'localForwarding', 'remoteForwarding', 'scp', 'sftp', 'shell', 'x11']
Test classes (2026): ['DoT', 'Encrypted-P2P', 'FTPS', 'HTTPS', 'SCP', 'SFTP', 'SSH-localForward', 'SSH-remoteForward', 'SSH-shell', 'SSH-x11', 'TLS-Telnet']


Based on standard mapping for traffic classification:
* `SSH-localForward` -> `localForwarding`
* `HTTPS` -> `HTTP` (or potentially browsing/lime, we will verify)
* `DoT` -> `DNS` (DNS over TLS is the modern encrypted DNS)
* `SSH-remoteForward` -> `remoteForwarding`
* `browsing` -> `HTTP` or `lime`
* `SFTP` -> `sftp`
* `streaming` -> `lime` / other
* `SSH-x11` -> `x11`
* `TLS-Telnet` -> `TELNET`
* `SSH-shell` -> `shell`
* `SCP` -> `scp`

Let's define a clean mapping dictionary to rename the classes in `test` to match the exact names used during training.

In [10]:
# Define the mapping from 2026 encrypted/updated classes to 2000 training classes
class_mapping = {
    'SSH-localForward': 'localForwarding',
    'SSH-remoteForward': 'remoteForwarding',
    'SSH-x11': 'x11',
    'SSH-shell': 'shell',
    'SFTP': 'sftp',
    'SCP': 'scp',
    'TLS-Telnet': 'TELNET',
    'DoT': 'DNS',
    'HTTPS': 'HTTP',
    'streaming': 'lime',
    'Encrypted-P2P': 'lime',
    'FTPS': 'FTP'
}

# Map the classes in the test set
test_mapped = test.copy()
test_mapped['class'] = test_mapped['class'].map(class_mapping)

print("Mapped test classes:", test_mapped['class'].unique())
print("Missing mappings:", test_mapped['class'].isna().sum())

Mapped test classes: ['localForwarding' 'HTTP' 'FTP' 'DNS' 'remoteForwarding' 'sftp' 'x11'
 'TELNET' 'lime' 'shell' 'scp']
Missing mappings: 0


In [11]:
# Preprocess the test dataset features
X_test_new = test_mapped.copy()
if 'id' in X_test_new.columns:
    X_test_new = X_test_new.drop(columns=['id'])

y_test_new = X_test_new.pop('class')

# Fill missing values using training medians
X_test_new[numeric_cols] = X_test_new[numeric_cols].fillna(X[numeric_cols].median())

# Scale numeric features using the exact same scaler used during training
X_test_scaled = X_test_new.copy()
X_test_scaled[numeric_cols] = scaler.transform(X_test_new[numeric_cols])

# Align columns with training features
X_test_encoded = pd.get_dummies(X_test_scaled)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

# Encode labels using the same label_encoder
y_test_encoded = label_encoder.transform(y_test_new)

# Evaluate all trained models on the test set
test_results = {}
for name, info in results.items():
    model = info['model']
    y_pred_new = model.predict(X_test_encoded)
    acc_new = accuracy_score(y_test_encoded, y_pred_new)
    test_results[name] = acc_new
    print(f"{name} Test Accuracy: {acc_new:.4f}")

print("\n--- Detailed Test Classification Reports ---")
for name, info in results.items():
    model = info['model']
    y_pred_new = model.predict(X_test_encoded)
    print(f"=== {name} on 2026 Test Data ===")
    print(classification_report(y_test_encoded, y_pred_new, target_names=label_encoder.classes_, zero_division=0))

Decision Tree Test Accuracy: 0.1005
Random Forest Test Accuracy: 0.3357
XGBoost Test Accuracy: 0.0386
Logistic Regression Test Accuracy: 0.1905
Linear SVM Test Accuracy: 0.2024
Neural Network Test Accuracy: 0.0447

--- Detailed Test Classification Reports ---
=== Decision Tree on 2026 Test Data ===
                  precision    recall  f1-score   support

             DNS       0.00      0.00      0.00       120
             FTP       0.00      0.00      0.00       579
            HTTP       0.08      0.06      0.07      1077
          TELNET       0.00      0.00      0.00        95
            lime       0.20      1.00      0.33        71
 localForwarding       0.00      0.00      0.00        22
remoteForwarding       0.00      0.00      0.00        40
             scp       0.54      0.48      0.51       329
            sftp       0.00      0.00      0.00       562
           shell       0.00      0.00      0.00        40
             x11       0.00      0.00      0.00        20

  

In [15]:
df = pd.read_csv("/content/train_traffic_features.csv")
df.head(10)

,id,min_fpktl,mean_fpktl,max_fpktl,std_fpktl,min_bpktl,mean_bpktl,max_bpktl,std_bpktl,min_fiat,...,mean_biat,max_biat,std_biat,duration,proto,total_fpackets,total_fvolume,total_bpackets,total_bvolume,class
0,1,52,138,1204,123,52,117,1172,138,6836,...,290777,2371821,589270,28787051,6,84,11552,100,11661,SSH-localForward
1,2,52,136,1204,128,52,118,1172,144,4,...,315652,2252872,629707,28408765,6,78,10600,91,10745,SSH-localForward
2,3,52,137,1204,119,52,115,1172,132,2,...,252521,2179968,547061,27272395,6,91,12508,109,12577,SSH-localForward
3,4,100,126,164,31,52,89,164,42,5897,...,395171,1914853,760018,2016872,6,5,628,6,536,SSH-localForward
4,5,52,134,1204,120,52,116,1172,134,2,...,284201,2315710,615149,29602740,6,91,12236,105,12145,SSH-localForward
5,6,52,138,1204,132,52,119,1172,147,4,...,339902,2386161,703100,28891756,6,72,9968,86,10261,SSH-localForward
6,7,52,128,1204,120,52,115,1172,136,2,...,295217,2386200,660839,29816988,6,94,12072,102,11765,SSH-localForward
7,8,52,137,1204,120,52,115,1172,134,6893,...,279201,2220294,545621,29316132,6,88,12080,106,12197,SSH-localForward
8,9,52,124,1204,120,52,115,1172,137,3,...,284542,2448070,625837,28738792,6,98,12184,102,11717,SSH-localForward
9,10,52,134,1204,119,52,114,1172,134,2,...,276372,2242863,604107,29295454,6,92,12288,107,12249,SSH-localForward


In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# Classifiers
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

# 1. Drop rows where target variable is missing
features = df.copy()
if 'id' in features.columns:
    features = features.drop(columns=['id'])

# Separate features and target
X = features.drop(columns=['class'])
y = features['class']

# Identify numeric columns for scaling
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Fill any missing values
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

# Encode target labels to integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Scale numerical features
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])

# Convert categorical features if any remain
X_encoded = pd.get_dummies(X_scaled)

# 2. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded,shuffle=True)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (2364, 22)
Testing shape: (591, 22)


In [24]:
# Initialize dictionaries to hold models and their performances
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=12, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1, eval_metric='mlogloss'),
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42, n_jobs=-1),
    'Linear SVM': LinearSVC(dual=False, random_state=42, max_iter=1000),
    # A single-layer MLP to balance speed and accuracy on large datasets
    'Neural Network': MLPClassifier(hidden_layer_sizes=(64,), max_iter=200, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results[name] = {
            'model': model,
            'accuracy': acc,
            'predictions': y_pred
        }
        print(f"{name} completed with Accuracy: {acc:.4f}\n")
    except Exception as e:
        print(f"Failed to train {name}: {e}\n")

Training Decision Tree...
Decision Tree completed with Accuracy: 0.9492

Training Random Forest...
Random Forest completed with Accuracy: 0.9695

Training XGBoost...
XGBoost completed with Accuracy: 0.9611

Training Logistic Regression...
Logistic Regression completed with Accuracy: 0.9069

Training Linear SVM...
Linear SVM completed with Accuracy: 0.9086

Training Neural Network...
Neural Network completed with Accuracy: 0.9408



/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [25]:
test_df = pd.read_csv("/content/test_traffic_features.csv")
test_df.head(10)

,id,min_fpktl,mean_fpktl,max_fpktl,std_fpktl,min_bpktl,mean_bpktl,max_bpktl,std_bpktl,min_fiat,...,mean_biat,max_biat,std_biat,duration,proto,total_fpackets,total_fvolume,total_bpackets,total_bvolume,class
0,1,52,134,1204,208,52,163,1172,245,3,...,130678,1610659,411591,3533934,6,29,3892,28,4557,SSH-localForward
1,2,52,144,1204,217,52,171,1172,252,13071,...,187841,2266040,493000,4696093,6,26,3736,26,4453,SSH-localForward
2,3,52,151,1204,224,52,176,1172,256,2844,...,160705,1620012,390480,3856995,6,24,3632,25,4401,SSH-localForward
3,4,52,151,1204,224,52,176,1172,256,4,...,127917,1520036,377251,3070058,6,24,3632,25,4401,SSH-localForward
4,5,52,81,1886,205,52,4083,11588,2804,0,...,6410,65213,13059,729435,6,89,7193,111,453181,HTTPS
5,6,52,52,52,0,68,4323,14252,3148,0,...,2754,40362,6567,295560,6,83,4316,106,458188,HTTPS
6,7,52,117,1818,226,52,2430,11412,1942,0,...,7336,109407,17342,1165012,6,93,10885,107,260056,HTTPS
7,8,52,57,287,35,52,3971,17092,3044,0,...,7809,252978,34829,452943,6,43,2471,59,234304,HTTPS
8,9,40,285,1854,594,52,604,2908,933,1,...,28254,93006,40264,285601,6,8,2278,8,4828,HTTPS
9,10,52,213,1822,429,52,799,2635,997,0,...,52716,536947,137459,824310,6,16,3404,15,11978,HTTPS


In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# Classifiers
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
test_df = pd.read_csv("/content/test_traffic_features.csv")
features = test_df.copy()
if 'id' in features.columns:
    features = features.drop(columns=['id'])

X_test_new = features.drop(columns=['class'])
y_test_new = features['class']


X_test_new[numeric_cols] = X_test_new[numeric_cols].fillna(X[numeric_cols].median())


X_test_scaled = X_test_new.copy()
X_test_scaled[numeric_cols] = scaler.transform(X_test_new[numeric_cols])


X_test_encoded = pd.get_dummies(X_test_scaled)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)


y_test_encoded = label_encoder.transform(y_test_new)

print("Test shape:", X_test_encoded.shape)
print("Test labels shape:", y_test_encoded.shape)

Test shape: (151, 22)
Test labels shape: (151,)


In [27]:
for name, model in models.items():
    print(f"Testing {name}...")
    try:
        y_pred = model.predict(X_test_encoded)
        acc = accuracy_score(y_test_encoded, y_pred)
        print(f"{name} Test Accuracy: {acc:.4f}\n")
    except Exception as e:
        print(f"Failed to test {name}: {e}\n")

Testing Decision Tree...
Decision Tree Test Accuracy: 0.8079

Testing Random Forest...
Random Forest Test Accuracy: 0.8940

Testing XGBoost...
XGBoost Test Accuracy: 0.8609

Testing Logistic Regression...
Logistic Regression Test Accuracy: 0.7351

Testing Linear SVM...
Linear SVM Test Accuracy: 0.7550

Testing Neural Network...
Neural Network Test Accuracy: 0.7881

